# 03 · Baselines y protocolo de evaluación**Frente de modelado · Sprint 1**Este notebook establece el **protocolo con el que se mide todo el proyecto** ydeja los dos puntos de referencia contra los que se compara cualquier modelopersonalizado.No implementa métricas: las importa de `src/evaluation/metrics.py`, que estátesteado. Si el baseline y el modelo se midieran con código distinto, lacomparación no valdría nada — y comparar es el criterio central de la Demo.**Requiere** haber ejecutado `python src/data/build_dataset.py` para tener lastablas en `data/processed/`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))
PROC = RAIZ / "data" / "processed"

from src.evaluation.metrics import evaluar, comparar

K = 10

## 1. El conjunto de evaluación`targets_train` es la última orden de cada usuario con sus productos revelados.Es lo que hay que predecir, y **nunca** puede usarse para construir variables.Se leen solo `user_id` y `product_id`. La tabla también trae `reordered` y`add_to_cart_order`, pero son de la orden objetivo: tenerlas a mano es tentar ausarlas como variables, y eso sería fuga de información.`evaluar()` espera un diccionario `{user_id: {product_id, ...}}`. Conjunto y nolista: el orden en que la persona compró no importa, solo si el producto estaba.

In [2]:
targets = pd.read_parquet(PROC / "targets_train.parquet",
                        columns=["user_id", "product_id"])

verdad = targets.groupby("user_id")["product_id"].apply(set).to_dict()

print(f"usuarios evaluables  : {len(verdad):,}")
print(f"productos a predecir : {sum(len(v) for v in verdad.values()):,}")
print()
print("ejemplo, usuario 1:", verdad[1])

usuarios evaluables  : 131,209
productos a predecir : 1,384,617

ejemplo, usuario 1: {196, 26405, 27845, 46149, 13032, 39657, 26088, 25133, 38928, 10258, 49235}


## 2. Baseline 1 · Popularidad globalLos K productos más comprados del histórico, los mismos para todos. No mira ala persona.Es el piso contra el que se compara todo, y el ejemplo más claro de por qué lacobertura de catálogo tiene que estar en la tabla: recomienda 10 productossobre 49.688, es decir el 0,02% del catálogo.*Detalle que aparece en el top-10:* `Organic Whole Milk` es el décimo productomás vendido del sitio, y es justamente el único producto **nuevo** de la últimaorden del usuario 1. La popularidad lo acierta; un modelo de pura recompra nopuede acertarlo nunca. Es la primera pista de que parte del descubrimiento sepuede capturar sin coocurrencia.

In [3]:
productos = pd.read_parquet(PROC / "productos.parquet",
                            columns=["product_id", "product_name",
                                    "cantidad_compras"])

top_global = productos.nlargest(K, "cantidad_compras")["product_id"].tolist()

recs_popularidad = {u: top_global for u in verdad}

productos.set_index("product_id").loc[top_global, ["product_name", "cantidad_compras"]]

,product_name,cantidad_compras
product_id,,
24852,Banana,472565
13176,Bag of Organic Bananas,379450
21137,Organic Strawberries,264683
21903,Organic Baby Spinach,241921
47209,Organic Hass Avocado,213584
47766,Organic Avocado,176815
47626,Large Lemon,152657
16797,Strawberries,142951
26209,Limes,140627


In [4]:
m_pop = evaluar(recs_popularidad, verdad, k=K)
m_pop

{'n_usuarios': 131209,
 'precision': 0.07252246416023292,
 'recall': 0.06984315126267857,
 'recall_macro': 0.06984315126267857,
 'recall_micro': 0.06872369760013058,
 'f1': 0.07115759545222561,
 'f1_micro': 0.07057199762525183,
 'hit_rate': 0.4583603258922787,
 'cobertura': 0.00020125583641925616,
 'usuarios_completos': 1.0,
 'aciertos_totales': 95156,
 'objetivos_totales': 1384617,
 'k': 10,
 'sin_recomendaciones': 0}

## 3. Baseline 2 · Recompra personalPara cada usuario, sus K productos ordenados por la **proporción de sus órdenes**que los contienen, con desempate por recencia.**Por qué la proporción y no el conteo crudo:** si alguien compró 8 veces unproducto en 80 órdenes y otro lo compró 4 veces en 5 órdenes, el segundo locompra mucho más seguido. La frecuencia absoluta favorece artificialmente a losusuarios con historial largo.**Por qué la recencia va ascendente:** `recencia_usuario_producto` cuentaórdenes desde la última compra, así que más chico es más reciente.Este es el baseline difícil de superar. Su límite: por construcción nunca puedeacertar un producto que la persona no haya comprado antes.

In [5]:
usuarios = pd.read_parquet(PROC / "usuarios.parquet",
                            columns=["user_id", "cantidad_ordenes_historicas",
                                    "segmento_usuario"])

inter = pd.read_parquet(
    PROC / "interacciones.parquet",
    columns=["user_id", "product_id",
            "freq_usuario_producto", "recencia_usuario_producto"],
)

# solo usuarios evaluables
inter = inter[inter["user_id"].isin(verdad)]

# proporcion de las ordenes del usuario que contienen el producto
n_ordenes = usuarios.set_index("user_id")["cantidad_ordenes_historicas"]
inter["ratio"] = inter["freq_usuario_producto"] / inter["user_id"].map(n_ordenes)

# mayor ratio primero; a igual ratio, el comprado mas recientemente
inter = inter.sort_values(["user_id", "ratio", "recencia_usuario_producto"],
                        ascending=[True, False, True])

top = inter.groupby("user_id").head(K)
recs_recompra = top.groupby("user_id")["product_id"].agg(list).to_dict()

sin_k = sum(1 for v in recs_recompra.values() if len(v) < K)
print(f"usuarios con recomendaciones : {len(recs_recompra):,}")
print(f"con menos de {K} productos     : {sin_k:,}")

usuarios con recomendaciones : 131,209
con menos de 10 productos     : 8,121


In [6]:
m_rec = evaluar(recs_recompra, verdad, k=K)
m_rec

{'n_usuarios': 131209,
 'precision': 0.27564801195039973,
 'recall': 0.32978418751152033,
 'recall_macro': 0.32978418751152033,
 'recall_micro': 0.26120941747790183,
 'f1': 0.30029574159094935,
 'f1_micro': 0.2682345542174215,
 'hit_rate': 0.8549489745368077,
 'cobertura': 0.7194493640315569,
 'usuarios_completos': 0.9381063798977204,
 'aciertos_totales': 361675,
 'objetivos_totales': 1384617,
 'k': 10,
 'sin_recomendaciones': 0}

## 4. ComparaciónLos dos baselines medidos con la misma función, sobre los mismos usuarios y conlas mismas métricas.El **lift** se calcula contra popularidad, que es la referencia que define elmarco de negocio.

In [7]:
tabla = pd.DataFrame(
    comparar({"popularidad": recs_popularidad,
            "recompra_personal": recs_recompra},
            verdad, k=K, referencia="popularidad")
)
tabla.round(4)

,modelo,precision,recall,recall_micro,f1,hit_rate,cobertura,n_usuarios,lift
0,popularidad,0.0725,0.0698,0.0687,0.0712,0.4584,0.0002,131209,0.0000
1,recompra_personal,0.2756,0.3298,0.2612,0.3003,0.8549,0.7194,131209,3.7218


## 5. Desglose por segmento, contra el techoEl **techo** es lo máximo que puede acertar un modelo que solo repite historial:la proporción de la próxima orden que es recompra.Con un detalle que no es obvio: **con K lugares no se pueden acertar más de Kproductos**, aunque el usuario vaya a repetir más. Por eso el `clip(upper=K)`.Sin ese recorte el techo de los usuarios heavy queda inflado —el 28,5% de ellosrepite más de 10 productos en su próxima orden— y parece que hay más margen delque realmente existe.La columna `pct_del_techo` es la que decide dónde poner el esfuerzo: dice quéproporción de lo alcanzable ya captura una regla de tres líneas.

In [8]:
segmentos = (usuarios.set_index("user_id")["segmento_usuario"]
             .loc[list(verdad)].to_dict())

# Techo: cuanto puede acertar como maximo un modelo que solo repite historial.
# El clip(upper=K) es clave: con K lugares no se puede acertar mas de K
# productos, aunque el usuario vaya a repetir mas. Sin ese recorte el techo
# de los usuarios heavy queda inflado y parece que hay mas margen del que hay.
tr = pd.read_parquet(PROC / "targets_train.parquet",
                     columns=["user_id", "reordered"])
g = tr.groupby("user_id")["reordered"].agg(["size", "sum"])
g.columns = ["items", "repetidos"]

g["techo"] = g["repetidos"].clip(upper=K) / g["items"]
techo_seg = g["techo"].groupby(pd.Series(segmentos)).mean()

det = evaluar(recs_recompra, verdad, k=K, segmentos=segmentos)

por_seg = pd.DataFrame(det["segmentos"]).T
por_seg["techo"] = techo_seg
por_seg["pct_del_techo"] = por_seg["recall"] / por_seg["techo"]

por_seg[["n_usuarios", "recall", "techo", "pct_del_techo",
         "hit_rate", "cobertura"]].round(4)

,n_usuarios,recall,techo,pct_del_techo,hit_rate,cobertura
heavy,41614.0,0.3344,0.6596,0.5070,0.8860,0.4530
medio,51519.0,0.3369,0.5491,0.6136,0.8597,0.5635
nuevo,38076.0,0.3151,0.4443,0.7092,0.8146,0.5500


## 6. Lectura### Los dos baselines| | popularidad | recompra personal ||---|---:|---:|| Recall@10 | 0,0698 | **0,3298** || Hit Rate@10 | 0,4584 | **0,8549** || Cobertura | 0,0002 | 0,7194 || Lift en Recall | — | **3,72** |### Cuatro conclusiones**1. Hit Rate y Recall no son intercambiables.** En popularidad, el 45,8% de losusuarios recibe al menos un producto que efectivamente compró, pero eso cubreapenas el 7% de su carrito. Los dos números salen de los mismos aciertos. El KPIde negocio —"% de pedidos en que el cliente suma al menos un productorecomendado"— es Hit Rate, no Recall.**2. La cobertura no mide descubrimiento.** La recompra personal cubre el 71,9%del catálogo y tiene descubrimiento **cero**: por construcción nunca recomiendaalgo que la persona no haya comprado. La cobertura es alta simplemente porquecada usuario recibe sus propios productos. Para validar la historia de usuariode descubrimiento hace falta otra métrica.**3. K = 10 queda confirmado empíricamente.** Precision 7,25% y Recall 6,98% enpopularidad: casi iguales, porque K está muy cerca del tamaño medio del carrito(10,6 productos). Con este K las dos métricas son comparables y ninguna estáinflada por la elección del corte.**4. El Recall es plano entre segmentos y la causa es K.** Heavy 0,3344, medio0,3369, nuevo 0,3151, pese a que un usuario heavy es mucho más predecible. Sucarrito tiene 11,1 productos con 8,3 repeticiones, y en diez lugares no entran.### Dónde poner el esfuerzo- **heavy** — captura el 50,7% de lo alcanzable. Quedan ~16 puntos de Recall  disponibles **solo ordenando mejor lo que ya compraron**. No necesita  candidatos nuevos.- **medio** — 61,4%.- **nuevo** — 70,9%. Queda poco por ganar con mejor ranking; lo que falta ahí  son productos que nunca compraron.Un modelo que aprenda a rankear ataca a heavy y medio, que son 93.000 de los131.209 usuarios. La generación de candidatos ataca a los nuevos.### Una lectura de negocioPara un cliente heavy el sistema **no puede** reconstruirle el carrito: no haylugar en diez sugerencias. Lo que puede hacer es que no se olvide lo importante.Para un cliente nuevo sí cubre casi todo lo repetible, y ahí el valor está endescubrir.Es la misma herramienta con dos propuestas de valor distintas según elsegmento.### Lo que sigue`04_modelo_personalizado.ipynb`: variables, modelos que aprenden, y la tabla decandidatos cuando esté disponible.